# Advanced Python Dictionaries: Problems with Complete Solutions

This notebook extends the topic **Creating Python Dictionaries** into advanced, practical exercises. It covers:

- dictionary literals, `dict()`, comprehensions, unpacking, unions, and `fromkeys`
- hashable keys and canonical representations
- duplicate-key policies and insertion order
- functions as dictionary keys and dispatch tables
- shallow versus deep copying
- nested dictionaries, indexing, grouping, graphs, tries, sparse matrices, and memoization
- validation, testing, performance, and JSON interoperability

Every problem includes a reference solution, tests, and a short best-practice explanation.

## How to Use This Notebook

For each problem:

1. Read the requirements and examples.
2. Hide or skip the solution cell and implement your own version.
3. Run the supplied tests.
4. Compare behavior, readability, edge-case handling, and complexity.

The solutions use only the Python standard library.

## Best-Practice Checklist

- Use a **literal** for a small, fixed mapping.
- Use `dict(iterable_of_pairs)` when data already arrives as key-value pairs.
- Use a **dictionary comprehension** when transforming or filtering.
- Use unpacking, `update`, or `|` only when overwrite precedence is intentional.
- Require keys to be hashable; convert mutable structures to stable immutable forms when necessary.
- Decide and document what duplicate keys mean: first wins, last wins, aggregate, or error.
- Remember that `dict(existing_dict)` and `.copy()` are shallow copies.
- Use `dict.fromkeys(keys, value)` only when sharing the exact same value is safe; immutable values such as `None`, integers, and strings are usually safe.
- Do not mutate a dictionary's size while iterating over it.
- Prefer small functions, type hints, meaningful names, and executable assertions.
- Modern Python preserves insertion order as a language guarantee (Python 3.7+).

## Setup and Test Helpers

In [1]:
from __future__ import annotations

from collections import deque
from collections.abc import Callable, Hashable, Iterable, Mapping, Sequence
from copy import deepcopy
from dataclasses import dataclass
from math import hypot
from pprint import pprint
from timeit import timeit
from typing import Any, Literal, TypeVar

K = TypeVar("K", bound=Hashable)
V = TypeVar("V")


def assert_raises(expected_exception, function, /, *args, **kwargs):
    """Return the raised exception, or fail if the call does not raise it."""
    try:
        function(*args, **kwargs)
    except expected_exception as exc:
        return exc
    except Exception as exc:  # pragma: no cover - diagnostic branch
        raise AssertionError(
            f"Expected {expected_exception.__name__}, got {type(exc).__name__}: {exc}"
        ) from exc
    raise AssertionError(f"Expected {expected_exception.__name__} to be raised")


print("Setup complete.")

Setup complete.


# Extended Construction Examples

These examples review the source topic before the advanced exercises.

### Example A — Equivalent Construction Mechanisms

In [2]:
keys = ["alpha", "beta", "gamma"]
values = [10, 20, 30]

examples = {
    "literal": {"alpha": 10, "beta": 20, "gamma": 30},
    "constructor_pairs": dict(zip(keys, values)),
    "comprehension": {key: value for key, value in zip(keys, values)},
    "keyword_arguments": dict(alpha=10, beta=20, gamma=30),
}

assert len({tuple(item.items()) for item in examples.values()}) == 1
examples

{'literal': {'alpha': 10, 'beta': 20, 'gamma': 30},
 'constructor_pairs': {'alpha': 10, 'beta': 20, 'gamma': 30},
 'comprehension': {'alpha': 10, 'beta': 20, 'gamma': 30},
 'keyword_arguments': {'alpha': 10, 'beta': 20, 'gamma': 30}}

### Example B — Hashable and Unhashable Keys

In [3]:
def is_hashable(value: Any) -> bool:
    try:
        hash(value)
    except TypeError:
        return False
    return True

candidates = [
    42,
    "python",
    (1, 2, 3),
    (1, [2, 3]),
    [1, 2, 3],
    frozenset({1, 2}),
]

hashability_report = [(repr(value), is_hashable(value)) for value in candidates]
hashability_report

[('42', True),
 ("'python'", True),
 ('(1, 2, 3)', True),
 ('(1, [2, 3])', False),
 ('[1, 2, 3]', False),
 ('frozenset({1, 2})', True)]

### Example C — Duplicate Keys and Insertion Order

When a key is assigned again, its value changes, but its original position is retained.

In [4]:
ordered = {"first": 1, "second": 2, "third": 3}
ordered["second"] = 200
ordered["fourth"] = 4

assert list(ordered) == ["first", "second", "third", "fourth"]
ordered

{'first': 1, 'second': 200, 'third': 3, 'fourth': 4}

### Example D — Safe and Unsafe `fromkeys`

All keys created by `fromkeys` reference the same value object. This is harmless for immutable values but dangerous for mutable containers.

In [5]:
safe_counters = dict.fromkeys(["ok", "warning", "error"], 0)
unsafe_buckets = dict.fromkeys(["north", "south", "east"], [])
unsafe_buckets["north"].append("shared")

safe_buckets = {region: [] for region in ["north", "south", "east"]}
safe_buckets["north"].append("independent")

{
    "safe_counters": safe_counters,
    "unsafe_buckets": unsafe_buckets,
    "safe_buckets": safe_buckets,
}

{'safe_counters': {'ok': 0, 'warning': 0, 'error': 0},
 'unsafe_buckets': {'north': ['shared'],
  'south': ['shared'],
  'east': ['shared']},
 'safe_buckets': {'north': ['independent'], 'south': [], 'east': []}}

# Problem 1 — Build a Validated Dictionary with a Duplicate-Key Policy

Implement `build_dict` from an iterable of two-item iterables.

Requirements:

- Validate that every item contains exactly two elements.
- Validate that every key is hashable.
- Support duplicate policies: `"first"`, `"last"`, and `"error"`.
- Preserve insertion order.
- Produce useful exception messages.

### Solution

In [6]:
DuplicatePolicy = Literal["first", "last", "error"]


def build_dict(
    entries: Iterable[Iterable[Any]],
    *,
    on_duplicate: DuplicatePolicy = "last",
) -> dict[Hashable, Any]:
    if on_duplicate not in {"first", "last", "error"}:
        raise ValueError("on_duplicate must be 'first', 'last', or 'error'")

    result: dict[Hashable, Any] = {}

    for position, entry in enumerate(entries):
        try:
            key, value = entry
        except (TypeError, ValueError) as exc:
            raise ValueError(
                f"Entry at position {position} must contain exactly two elements"
            ) from exc

        try:
            hash(key)
        except TypeError as exc:
            raise TypeError(
                f"Key at position {position} is not hashable: {key!r}"
            ) from exc

        if key in result:
            if on_duplicate == "first":
                continue
            if on_duplicate == "error":
                raise KeyError(f"Duplicate key at position {position}: {key!r}")

        result[key] = value

    return result


entries = [("a", 1), ["b", 2], ("a", 99), ((1, 2), "tuple key")]

assert build_dict(entries, on_duplicate="first")["a"] == 1
assert build_dict(entries, on_duplicate="last")["a"] == 99
assert_raises(KeyError, build_dict, entries, on_duplicate="error")
assert_raises(ValueError, build_dict, [("a", 1, 2)])
assert_raises(TypeError, build_dict, [([1, 2], "bad key")])

build_dict(entries, on_duplicate="last")

{'a': 99, 'b': 2, (1, 2): 'tuple key'}

**Best practice:** duplicate-key behavior should be explicit at system boundaries. Silent overwrites are convenient, but they can hide malformed input.

# Problem 2 — Convert Nested Mutable Data into a Hashable Canonical Key

A cache needs to use nested configuration objects as dictionary keys. Lists, sets, and dictionaries are unhashable.

Implement `freeze` so logically equivalent nested values produce the same immutable key. Preserve the difference between lists and tuples, and make dictionary order irrelevant.

### Solution

In [7]:
def freeze(value: Any) -> Hashable:
    """Recursively convert common containers into a stable hashable representation."""
    if isinstance(value, Mapping):
        frozen_items = [(freeze(key), freeze(item)) for key, item in value.items()]
        frozen_items.sort(key=repr)
        return ("mapping", tuple(frozen_items))

    if isinstance(value, list):
        return ("list", tuple(freeze(item) for item in value))

    if isinstance(value, tuple):
        return ("tuple", tuple(freeze(item) for item in value))

    if isinstance(value, set):
        return ("set", frozenset(freeze(item) for item in value))

    if isinstance(value, frozenset):
        return ("frozenset", frozenset(freeze(item) for item in value))

    try:
        hash(value)
    except TypeError as exc:
        raise TypeError(f"Unsupported unhashable value: {value!r}") from exc

    return value


config_a = {
    "model": "x1",
    "features": ["fast", "safe"],
    "limits": {"timeout": 5, "retries": 2},
}
config_b = {
    "limits": {"retries": 2, "timeout": 5},
    "features": ["fast", "safe"],
    "model": "x1",
}

key_a = freeze(config_a)
key_b = freeze(config_b)
cache = {key_a: "compiled-result"}

assert key_a == key_b
assert cache[key_b] == "compiled-result"
assert freeze([1, 2]) != freeze((1, 2))

{"same_key": key_a == key_b, "cached_value": cache[key_b]}

{'same_key': True, 'cached_value': 'compiled-result'}

**Best practice:** canonicalization rules are part of the data model. Document whether order, container type, floating-point precision, and custom objects matter.

# Problem 3 — Execute a Plan Whose Dictionary Keys Are Functions

Functions are hashable. Build an execution engine where each key is a callable and each value is a sequence of calls represented as `(args, kwargs)`.

### Solution

In [8]:
def add(a: float, b: float) -> float:
    return a + b


def divide(a: float, b: float) -> float:
    return a / b


def power(base: float, exponent: float = 2) -> float:
    return base**exponent


CallSpec = tuple[tuple[Any, ...], dict[str, Any]]


def execute_plan(
    plan: Mapping[Callable[..., Any], Sequence[CallSpec]],
) -> dict[Callable[..., Any], list[Any]]:
    results: dict[Callable[..., Any], list[Any]] = {}

    for function, calls in plan.items():
        if not callable(function):
            raise TypeError(f"Plan key is not callable: {function!r}")
        results[function] = [function(*args, **kwargs) for args, kwargs in calls]

    return results


plan = {
    add: [((2, 3), {}), ((10, -4), {})],
    divide: [((10, 2), {}), ((7, 2), {})],
    power: [((5,), {}), ((2,), {"exponent": 8})],
}

execution = execute_plan(plan)
readable_execution = {function.__name__: values for function, values in execution.items()}

assert readable_execution == {
    "add": [5, 6],
    "divide": [5.0, 3.5],
    "power": [25, 256],
}
readable_execution

{'add': [5, 6], 'divide': [5.0, 3.5], 'power': [25, 256]}

**Best practice:** callable keys are useful for in-process registries. For persisted data or cross-process communication, stable string identifiers are usually safer than function objects.

# Problem 4 — Merge Configuration Layers and Track Provenance

Merge named configuration layers from lowest to highest precedence. Return both the merged dictionary and a dictionary showing which layer supplied each final value.

Replacing a value must not move the key to the end.

### Solution

In [9]:
def merge_layers(
    *layers: tuple[str, Mapping[str, Any]],
) -> tuple[dict[str, Any], dict[str, str]]:
    merged: dict[str, Any] = {}
    provenance: dict[str, str] = {}

    for layer_name, layer in layers:
        for key, value in layer.items():
            merged[key] = value
            provenance[key] = layer_name

    return merged, provenance


defaults = {"host": "localhost", "port": 8000, "debug": False}
environment = {"port": 8080, "workers": 4}
user = {"debug": True, "theme": "dark"}

merged, provenance = merge_layers(
    ("defaults", defaults),
    ("environment", environment),
    ("user", user),
)

assert merged == {
    "host": "localhost",
    "port": 8080,
    "debug": True,
    "workers": 4,
    "theme": "dark",
}
assert list(merged) == ["host", "port", "debug", "workers", "theme"]
assert provenance["port"] == "environment"
assert provenance["debug"] == "user"

{"merged": merged, "provenance": provenance}

{'merged': {'host': 'localhost',
  'port': 8080,
  'debug': True,
  'workers': 4,
  'theme': 'dark'},
 'provenance': {'host': 'defaults',
  'port': 'environment',
  'debug': 'user',
  'workers': 'environment',
  'theme': 'user'}}

**Best practice:** make precedence visible. Configuration bugs are easier to diagnose when final values can be traced to their source layer.

# Problem 5 — Demonstrate and Control Shallow versus Deep Copying

Implement `clone_mapping(mapping, deep=False)`. Show that a shallow copy creates a new outer dictionary but still shares nested mutable objects.

### Solution

In [10]:
def clone_mapping(mapping: Mapping[K, V], *, deep: bool = False) -> dict[K, V]:
    return deepcopy(mapping) if deep else dict(mapping)


original = {
    "service": {"host": "localhost", "ports": [8000, 8001]},
    "enabled": True,
}
shallow = clone_mapping(original)
deep = clone_mapping(original, deep=True)

shallow["service"]["ports"].append(9000)

assert shallow is not original
assert shallow["service"] is original["service"]
assert deep["service"] is not original["service"]
assert original["service"]["ports"] == [8000, 8001, 9000]
assert deep["service"]["ports"] == [8000, 8001]

{
    "outer_objects_differ": shallow is not original,
    "shallow_nested_shared": shallow["service"] is original["service"],
    "deep_nested_shared": deep["service"] is original["service"],
}

{'outer_objects_differ': True,
 'shallow_nested_shared': True,
 'deep_nested_shared': False}

**Best practice:** choose copy depth based on ownership. Deep copying can be expensive and may be inappropriate for objects such as files, locks, database connections, or shared caches.

# Problem 6 — Repair the Mutable-Default `fromkeys` Bug

Create independent list buckets for a set of labels. First reproduce the bug caused by `dict.fromkeys(labels, [])`, then implement a safe constructor.

### Solution

In [11]:
def make_buckets(labels: Iterable[K]) -> dict[K, list[Any]]:
    return {label: [] for label in labels}


labels = ["critical", "warning", "info"]

broken = dict.fromkeys(labels, [])
broken["critical"].append("disk full")

fixed = make_buckets(labels)
fixed["critical"].append("disk full")

assert broken == {
    "critical": ["disk full"],
    "warning": ["disk full"],
    "info": ["disk full"],
}
assert fixed == {
    "critical": ["disk full"],
    "warning": [],
    "info": [],
}

{"broken": broken, "fixed": fixed}

{'broken': {'critical': ['disk full'],
  'warning': ['disk full'],
  'info': ['disk full']},
 'fixed': {'critical': ['disk full'], 'warning': [], 'info': []}}

**Best practice:** use `fromkeys` with a mutable value only when shared state is intentional and clearly documented.

# Problem 7 — Build a Filtered Coordinate-Distance Map with a Comprehension

Create a dictionary whose keys are `(x, y)` coordinate tuples and whose values are Euclidean distances from the origin.

Include only points whose distance lies in the inclusive interval `[minimum, maximum]`, and round each value to three decimals.

### Solution

In [12]:
def build_distance_map(
    x_values: Iterable[float],
    y_values: Iterable[float],
    *,
    minimum: float = 0.0,
    maximum: float = float("inf"),
) -> dict[tuple[float, float], float]:
    if minimum < 0 or maximum < minimum:
        raise ValueError("Require 0 <= minimum <= maximum")

    return {
        (x, y): round(distance, 3)
        for x in x_values
        for y in y_values
        if minimum <= (distance := hypot(x, y)) <= maximum
    }


distance_map = build_distance_map(range(-3, 4), range(-3, 4), minimum=2, maximum=3)

assert distance_map[(0, 2)] == 2.0
assert (0, 0) not in distance_map
assert all(2 <= distance <= 3 for distance in distance_map.values())

{"point_count": len(distance_map), "sample": list(distance_map.items())[:8]}

{'point_count': 20,
 'sample': [((-3, 0), 3.0),
  ((-2, -2), 2.828),
  ((-2, -1), 2.236),
  ((-2, 0), 2.0),
  ((-2, 1), 2.236),
  ((-2, 2), 2.828),
  ((-1, -2), 2.236),
  ((-1, 2), 2.236)]}

**Best practice:** use the assignment expression only when it prevents meaningful repeated work and remains readable.

# Problem 8 — Invert a Dictionary Without Losing Collisions

A naive inversion `{value: key for key, value in mapping.items()}` loses information when several keys share one value.

Implement a multimap inversion where each original value maps to a list of original keys in insertion order.

### Solution

In [13]:
def invert_multimap(mapping: Mapping[K, V]) -> dict[V, list[K]]:
    inverted: dict[V, list[K]] = {}

    for key, value in mapping.items():
        try:
            hash(value)
        except TypeError as exc:
            raise TypeError(f"Cannot use unhashable value as inverted key: {value!r}") from exc

        inverted.setdefault(value, []).append(key)

    return inverted


roles = {
    "alice": "admin",
    "bob": "viewer",
    "cara": "admin",
    "dan": "editor",
    "erin": "viewer",
}

by_role = invert_multimap(roles)

assert by_role == {
    "admin": ["alice", "cara"],
    "viewer": ["bob", "erin"],
    "editor": ["dan"],
}
by_role

{'admin': ['alice', 'cara'], 'viewer': ['bob', 'erin'], 'editor': ['dan']}

**Best practice:** when inversion is not one-to-one, represent that fact explicitly rather than silently overwriting data.

# Problem 9 — Index Records with Explicit Duplicate Handling

Build an index from a sequence of record mappings. The selected field becomes the dictionary key.

Support `"first"`, `"last"`, and `"error"` duplicate policies. Copy each record so later external mutation does not alter the index entry's outer dictionary.

### Solution

In [14]:
def index_records(
    records: Iterable[Mapping[str, Any]],
    field: str,
    *,
    on_duplicate: DuplicatePolicy = "error",
) -> dict[Hashable, dict[str, Any]]:
    if on_duplicate not in {"first", "last", "error"}:
        raise ValueError("Invalid duplicate policy")

    index: dict[Hashable, dict[str, Any]] = {}

    for position, record in enumerate(records):
        if field not in record:
            raise KeyError(f"Record {position} is missing field {field!r}")

        key = record[field]
        try:
            hash(key)
        except TypeError as exc:
            raise TypeError(f"Record {position} has an unhashable index value") from exc

        if key in index:
            if on_duplicate == "first":
                continue
            if on_duplicate == "error":
                raise KeyError(f"Duplicate index value: {key!r}")

        index[key] = dict(record)

    return index


users = [
    {"id": 101, "name": "Asha", "active": True},
    {"id": 102, "name": "Boris", "active": False},
    {"id": 103, "name": "Chen", "active": True},
]

users_by_id = index_records(users, "id")
users[0]["name"] = "MUTATED OUTSIDE"

assert users_by_id[101]["name"] == "Asha"
assert users_by_id[103]["active"] is True
assert_raises(KeyError, index_records, users + [{"id": 101}], "id")

users_by_id

{101: {'id': 101, 'name': 'Asha', 'active': True},
 102: {'id': 102, 'name': 'Boris', 'active': False},
 103: {'id': 103, 'name': 'Chen', 'active': True}}

**Best practice:** an index is a contract. Validate missing keys, duplicate identifiers, hashability, and ownership of stored records.

# Problem 10 — Aggregate Transactions into Nested Dictionaries

Transform transaction records into this structure:

```python
customer -> category -> {"units": total_units, "revenue": total_revenue}
```

Do not assume customers or categories are known in advance.

### Solution

In [15]:
def aggregate_transactions(
    transactions: Iterable[Mapping[str, Any]],
) -> dict[str, dict[str, dict[str, float | int]]]:
    summary: dict[str, dict[str, dict[str, float | int]]] = {}

    for position, transaction in enumerate(transactions):
        try:
            customer = str(transaction["customer"])
            category = str(transaction["category"])
            units = int(transaction["units"])
            unit_price = float(transaction["unit_price"])
        except (KeyError, TypeError, ValueError) as exc:
            raise ValueError(f"Invalid transaction at position {position}") from exc

        if units < 0 or unit_price < 0:
            raise ValueError("units and unit_price must be non-negative")

        customer_summary = summary.setdefault(customer, {})
        bucket = customer_summary.setdefault(
            category,
            {"units": 0, "revenue": 0.0},
        )
        bucket["units"] += units
        bucket["revenue"] = round(bucket["revenue"] + units * unit_price, 2)

    return summary


transactions = [
    {"customer": "A", "category": "books", "units": 2, "unit_price": 12.50},
    {"customer": "A", "category": "games", "units": 1, "unit_price": 40.00},
    {"customer": "B", "category": "books", "units": 3, "unit_price": 10.00},
    {"customer": "A", "category": "books", "units": 1, "unit_price": 15.00},
]

transaction_summary = aggregate_transactions(transactions)

assert transaction_summary["A"]["books"] == {"units": 3, "revenue": 40.0}
assert transaction_summary["A"]["games"] == {"units": 1, "revenue": 40.0}
assert transaction_summary["B"]["books"] == {"units": 3, "revenue": 30.0}

transaction_summary

{'A': {'books': {'units': 3, 'revenue': 40.0},
  'games': {'units': 1, 'revenue': 40.0}},
 'B': {'books': {'units': 3, 'revenue': 30.0}}}

**Best practice:** nested `setdefault` is appropriate when the initialization is simple. For deeper or more complex aggregation, a helper function or `defaultdict` can improve readability.

# Problem 11 — Represent and Manipulate a Sparse Matrix with Tuple Keys

Use `(row, column)` tuples as keys and store only nonzero values.

Implement:

- `dense_to_sparse`
- `sparse_to_dense`
- `transpose_sparse`
- `add_sparse`

### Solution

In [16]:
SparseMatrix = dict[tuple[int, int], float]


def dense_to_sparse(matrix: Sequence[Sequence[float]]) -> tuple[SparseMatrix, tuple[int, int]]:
    rows = len(matrix)
    columns = len(matrix[0]) if rows else 0

    if any(len(row) != columns for row in matrix):
        raise ValueError("Dense matrix must be rectangular")

    sparse = {
        (row_index, column_index): float(value)
        for row_index, row in enumerate(matrix)
        for column_index, value in enumerate(row)
        if value != 0
    }
    return sparse, (rows, columns)


def sparse_to_dense(sparse: Mapping[tuple[int, int], float], shape: tuple[int, int]) -> list[list[float]]:
    rows, columns = shape
    if rows < 0 or columns < 0:
        raise ValueError("Shape dimensions must be non-negative")

    dense = [[0.0 for _ in range(columns)] for _ in range(rows)]
    for (row, column), value in sparse.items():
        if not (0 <= row < rows and 0 <= column < columns):
            raise IndexError(f"Coordinate {(row, column)} is outside shape {shape}")
        dense[row][column] = float(value)
    return dense


def transpose_sparse(
    sparse: Mapping[tuple[int, int], float],
    shape: tuple[int, int],
) -> tuple[SparseMatrix, tuple[int, int]]:
    rows, columns = shape
    return {(column, row): value for (row, column), value in sparse.items()}, (columns, rows)


def add_sparse(left: Mapping[tuple[int, int], float], right: Mapping[tuple[int, int], float]) -> SparseMatrix:
    result = dict(left)
    for coordinate, value in right.items():
        result[coordinate] = result.get(coordinate, 0.0) + value
        if result[coordinate] == 0:
            del result[coordinate]
    return result


dense = [
    [0, 5, 0],
    [2, 0, 0],
    [0, 0, 9],
]
sparse, shape = dense_to_sparse(dense)
transposed, transposed_shape = transpose_sparse(sparse, shape)
sum_matrix = add_sparse(sparse, {(0, 1): -5, (1, 2): 7})

assert sparse == {(0, 1): 5.0, (1, 0): 2.0, (2, 2): 9.0}
assert sparse_to_dense(sparse, shape) == [
    [0.0, 5.0, 0.0],
    [2.0, 0.0, 0.0],
    [0.0, 0.0, 9.0],
]
assert transposed[(1, 0)] == 5.0
assert transposed_shape == (3, 3)
assert (0, 1) not in sum_matrix
assert sum_matrix[(1, 2)] == 7

{"sparse": sparse, "transposed": transposed, "sum": sum_matrix}

{'sparse': {(0, 1): 5.0, (1, 0): 2.0, (2, 2): 9.0},
 'transposed': {(1, 0): 5.0, (0, 1): 2.0, (2, 2): 9.0},
 'sum': {(1, 0): 2.0, (2, 2): 9.0, (1, 2): 7.0}}

**Best practice:** tuple keys are ideal for multidimensional coordinates because tuples are immutable, hashable, compact, and naturally unpacked.

# Problem 12 — Use a Dictionary as a Memoization Cache

Compute the length of the longest common subsequence of two strings. Use `(i, j)` tuple keys to memoize subproblems.

Return both the answer and the cache so its structure is inspectable.

### Solution

In [17]:
def longest_common_subsequence_length(
    left: str,
    right: str,
) -> tuple[int, dict[tuple[int, int], int]]:
    cache: dict[tuple[int, int], int] = {}

    def solve(i: int, j: int) -> int:
        key = (i, j)
        if key in cache:
            return cache[key]

        if i == len(left) or j == len(right):
            answer = 0
        elif left[i] == right[j]:
            answer = 1 + solve(i + 1, j + 1)
        else:
            answer = max(solve(i + 1, j), solve(i, j + 1))

        cache[key] = answer
        return answer

    return solve(0, 0), cache


length, lcs_cache = longest_common_subsequence_length("DICTIONARY", "ACTION")

assert length == 5  # one valid LCS is "CTION"
assert lcs_cache[(0, 0)] == length
assert all(isinstance(key, tuple) and len(key) == 2 for key in lcs_cache)

{"length": length, "cached_subproblems": len(lcs_cache)}

{'length': 5, 'cached_subproblems': 72}

**Best practice:** cache keys must contain every input dimension that affects the result. Missing one dimension creates incorrect cache hits.

# Problem 13 — Build a Trie from Words Using Nested Dictionaries

A trie stores each character in a nested dictionary. Implement construction, exact-word lookup, and prefix lookup.

### Solution

In [18]:
END_OF_WORD = "<END>"


def build_trie(words: Iterable[str]) -> dict[str, Any]:
    root: dict[str, Any] = {}

    for word in words:
        node = root
        for character in word:
            node = node.setdefault(character, {})
        node[END_OF_WORD] = True

    return root


def trie_contains(trie: Mapping[str, Any], word: str) -> bool:
    node: Mapping[str, Any] = trie
    for character in word:
        child = node.get(character)
        if not isinstance(child, Mapping):
            return False
        node = child
    return node.get(END_OF_WORD) is True


def trie_has_prefix(trie: Mapping[str, Any], prefix: str) -> bool:
    node: Mapping[str, Any] = trie
    for character in prefix:
        child = node.get(character)
        if not isinstance(child, Mapping):
            return False
        node = child
    return True


words = ["cat", "car", "cart", "dog", "door"]
trie = build_trie(words)

assert trie_contains(trie, "car")
assert trie_contains(trie, "cart")
assert not trie_contains(trie, "ca")
assert trie_has_prefix(trie, "ca")
assert trie_has_prefix(trie, "doo")
assert not trie_has_prefix(trie, "z")

{
    "contains_car": trie_contains(trie, "car"),
    "contains_ca": trie_contains(trie, "ca"),
    "has_prefix_doo": trie_has_prefix(trie, "doo"),
    "root_keys": list(trie),
}

{'contains_car': True,
 'contains_ca': False,
 'has_prefix_doo': True,
 'root_keys': ['c', 'd']}

**Best practice:** reserve terminal markers carefully. In production, a unique sentinel object avoids collisions with ordinary keys, although such sentinels need custom serialization.

# Problem 14 — Build an Order-Preserving Graph Adjacency Dictionary

Convert an edge list into `node -> list of neighbors`.

Requirements:

- support directed and undirected graphs
- include nodes with no outgoing edges
- remove duplicate edges
- preserve first-seen node and neighbor order
- implement breadth-first search using the result

### Solution

In [19]:
def build_adjacency(
    edges: Iterable[tuple[K, K]],
    *,
    directed: bool = False,
) -> dict[K, list[K]]:
    adjacency: dict[K, list[K]] = {}
    seen_neighbors: dict[K, set[K]] = {}

    def ensure_node(node: K) -> None:
        adjacency.setdefault(node, [])
        seen_neighbors.setdefault(node, set())

    def add_edge(source: K, target: K) -> None:
        ensure_node(source)
        ensure_node(target)
        if target not in seen_neighbors[source]:
            adjacency[source].append(target)
            seen_neighbors[source].add(target)

    for source, target in edges:
        add_edge(source, target)
        if not directed:
            add_edge(target, source)

    return adjacency


def breadth_first_order(adjacency: Mapping[K, Sequence[K]], start: K) -> list[K]:
    if start not in adjacency:
        return []

    visited = {start}
    queue = deque([start])
    order: list[K] = []

    while queue:
        node = queue.popleft()
        order.append(node)
        for neighbor in adjacency.get(node, []):
            if neighbor not in visited:
                visited.add(neighbor)
                queue.append(neighbor)

    return order


edges = [
    ("A", "B"),
    ("A", "C"),
    ("B", "D"),
    ("A", "B"),  # duplicate
    ("C", "D"),
    ("E", "E"),  # isolated component with self-loop
]

adjacency = build_adjacency(edges)

assert adjacency["A"] == ["B", "C"]
assert adjacency["B"] == ["A", "D"]
assert adjacency["E"] == ["E"]
assert breadth_first_order(adjacency, "A") == ["A", "B", "C", "D"]

{"adjacency": adjacency, "bfs_from_A": breadth_first_order(adjacency, "A")}

{'adjacency': {'A': ['B', 'C'],
  'B': ['A', 'D'],
  'C': ['A', 'D'],
  'D': ['B', 'C'],
  'E': ['E']},
 'bfs_from_A': ['A', 'B', 'C', 'D']}

**Best practice:** a dictionary of lists preserves traversal order; a dictionary of sets provides faster membership but does not express a deliberate neighbor order.

# Problem 15 — Convert Dotted Paths into Nested Dictionaries

Convert flat configuration keys such as `"database.host"` into nested dictionaries. Detect conflicts such as assigning both `"a"` and `"a.b"`.

Also implement the reverse operation.

### Solution

In [20]:
def unflatten_mapping(flat: Mapping[str, Any], *, separator: str = ".") -> dict[str, Any]:
    if not separator:
        raise ValueError("separator must not be empty")

    nested: dict[str, Any] = {}

    for compound_key, value in flat.items():
        parts = compound_key.split(separator)
        if any(part == "" for part in parts):
            raise ValueError(f"Invalid empty path segment in {compound_key!r}")

        cursor = nested
        for part in parts[:-1]:
            existing = cursor.get(part)
            if existing is None:
                cursor[part] = {}
            elif not isinstance(existing, dict):
                raise ValueError(f"Path conflict at {part!r} in {compound_key!r}")
            cursor = cursor[part]

        leaf = parts[-1]
        if leaf in cursor:
            raise ValueError(f"Duplicate or conflicting path: {compound_key!r}")
        cursor[leaf] = value

    return nested


def flatten_mapping(
    nested: Mapping[str, Any],
    *,
    separator: str = ".",
    prefix: str = "",
) -> dict[str, Any]:
    flat: dict[str, Any] = {}

    for key, value in nested.items():
        compound_key = f"{prefix}{separator}{key}" if prefix else key
        if isinstance(value, Mapping):
            flat.update(
                flatten_mapping(value, separator=separator, prefix=compound_key)
            )
        else:
            flat[compound_key] = value

    return flat


flat_config = {
    "database.host": "db.internal",
    "database.port": 5432,
    "features.search.enabled": True,
    "features.export.enabled": False,
}

nested_config = unflatten_mapping(flat_config)
round_trip = flatten_mapping(nested_config)

assert round_trip == flat_config
assert nested_config["database"]["port"] == 5432
assert_raises(ValueError, unflatten_mapping, {"a": 1, "a.b": 2})

nested_config

{'database': {'host': 'db.internal', 'port': 5432},
 'features': {'search': {'enabled': True}, 'export': {'enabled': False}}}

**Best practice:** path syntax needs a clear escaping strategy if real keys may contain the separator.

# Problem 16 — Filter a Dictionary Without Mutating It During Iteration

Given an account dictionary, create a new dictionary containing only active accounts with non-negative balances. Normalize names to title case.

Do not delete keys from the original while iterating over it.

### Solution

In [21]:
accounts = {
    "u1": {"name": "ada lovelace", "active": True, "balance": 125.50},
    "u2": {"name": "grace hopper", "active": False, "balance": 88.00},
    "u3": {"name": "alan turing", "active": True, "balance": -5.00},
    "u4": {"name": "katherine johnson", "active": True, "balance": 300.00},
}

active_accounts = {
    account_id: {
        **details,
        "name": details["name"].title(),
    }
    for account_id, details in accounts.items()
    if details["active"] and details["balance"] >= 0
}

assert list(active_accounts) == ["u1", "u4"]
assert active_accounts["u1"]["name"] == "Ada Lovelace"
assert accounts["u1"]["name"] == "ada lovelace"

active_accounts

{'u1': {'name': 'Ada Lovelace', 'active': True, 'balance': 125.5},
 'u4': {'name': 'Katherine Johnson', 'active': True, 'balance': 300.0}}

**Best practice:** comprehensions are often the clearest way to filter and transform mappings while preserving the original.

# Problem 17 — Choose Between `dict(zip(...))` and a Comprehension

Construct a mapping for 10,000 keys in two ways and benchmark them. Then use a comprehension for a transformed and filtered result.

Timing values are machine-dependent; the goal is to practice measurement, not declare a universal winner.

### Solution

In [22]:
benchmark_keys = [f"k{i}" for i in range(10_000)]
benchmark_values = list(range(10_000))


def with_constructor() -> dict[str, int]:
    return dict(zip(benchmark_keys, benchmark_values))


def with_comprehension() -> dict[str, int]:
    return {key: value for key, value in zip(benchmark_keys, benchmark_values)}


constructor_time = timeit(with_constructor, number=100)
comprehension_time = timeit(with_comprehension, number=100)

transformed = {
    key: value * value
    for key, value in zip(benchmark_keys, benchmark_values)
    if value % 2 == 0
}

assert with_constructor() == with_comprehension()
assert len(transformed) == 5_000

{
    "dict_zip_seconds": round(constructor_time, 4),
    "comprehension_seconds": round(comprehension_time, 4),
    "transformed_entries": len(transformed),
}

{'dict_zip_seconds': 0.0888,
 'comprehension_seconds': 0.0998,
 'transformed_entries': 5000}

**Best practice:** prefer `dict(zip(keys, values))` for direct pairing and a comprehension for transformation or filtering. Optimize only after measuring representative workloads.

# Problem 18 — Capstone: Build an Event Router and Metrics Report

Create a decorator-based handler registry and process event dictionaries.

The final report must contain:

- processed results grouped by event type
- a count for every registered handler, including zero counts
- structured errors for unknown or failed events

Use `dict.fromkeys` safely for integer counters.

### Solution

In [23]:
Handler = Callable[[Mapping[str, Any]], Any]
handlers: dict[str, Handler] = {}


def register(event_type: str):
    def decorator(function: Handler) -> Handler:
        if event_type in handlers:
            raise KeyError(f"Handler already registered for {event_type!r}")
        handlers[event_type] = function
        return function
    return decorator


@register("login")
def handle_login(payload: Mapping[str, Any]) -> str:
    return f"welcome:{payload['user']}"


@register("purchase")
def handle_purchase(payload: Mapping[str, Any]) -> float:
    amount = float(payload["amount"])
    if amount < 0:
        raise ValueError("amount must be non-negative")
    return round(amount, 2)


@register("logout")
def handle_logout(payload: Mapping[str, Any]) -> str:
    return f"goodbye:{payload['user']}"


def process_events(events: Iterable[Mapping[str, Any]]) -> dict[str, Any]:
    counts = dict.fromkeys(handlers, 0)  # integers are immutable: safe shared initial value
    results = {event_type: [] for event_type in handlers}
    errors: list[dict[str, Any]] = []

    for position, event in enumerate(events):
        event_type = event.get("type")
        payload = event.get("payload", {})
        handler = handlers.get(event_type)

        if handler is None:
            errors.append(
                {
                    "position": position,
                    "type": event_type,
                    "error": "unknown event type",
                }
            )
            continue

        try:
            result = handler(payload)
        except Exception as exc:
            errors.append(
                {
                    "position": position,
                    "type": event_type,
                    "error": f"{type(exc).__name__}: {exc}",
                }
            )
            continue

        counts[event_type] += 1
        results[event_type].append(result)

    return {"results": results, "counts": counts, "errors": errors}


events = [
    {"type": "login", "payload": {"user": "Asha"}},
    {"type": "purchase", "payload": {"amount": 19.995}},
    {"type": "unknown", "payload": {}},
    {"type": "purchase", "payload": {"amount": -3}},
    {"type": "logout", "payload": {"user": "Asha"}},
]

report = process_events(events)

assert report["counts"] == {"login": 1, "purchase": 1, "logout": 1}
assert report["results"]["purchase"] == [20.0]
assert len(report["errors"]) == 2
assert list(report["counts"]) == ["login", "purchase", "logout"]

report

{'results': {'login': ['welcome:Asha'],
  'purchase': [20.0],
  'logout': ['goodbye:Asha']},
 'counts': {'login': 1, 'purchase': 1, 'logout': 1},
 'errors': [{'position': 2, 'type': 'unknown', 'error': 'unknown event type'},
  {'position': 3,
   'type': 'purchase',
   'error': 'ValueError: amount must be non-negative'}]}

**Best practice:** registries should reject duplicate registrations, isolate handler failures, and return structured diagnostics rather than silently dropping events.

# Problem 19 — Compute a Recursive Difference Between Nested Dictionaries

Implement `dictionary_diff(before, after)`.

Return a dictionary with three sections:

- `added`: paths present only in `after`
- `removed`: paths present only in `before`
- `changed`: paths whose leaf values differ, stored as `(old, new)`

Use tuple paths so the result keys are hashable and unambiguous.

### Solution

In [24]:
def dictionary_diff(
    before: Mapping[str, Any],
    after: Mapping[str, Any],
) -> dict[str, dict[tuple[str, ...], Any]]:
    result: dict[str, dict[tuple[str, ...], Any]] = {
        "added": {},
        "removed": {},
        "changed": {},
    }

    def walk(left: Any, right: Any, path: tuple[str, ...]) -> None:
        if isinstance(left, Mapping) and isinstance(right, Mapping):
            all_keys = list(left)
            all_keys.extend(key for key in right if key not in left)

            for key in all_keys:
                next_path = path + (str(key),)
                if key not in left:
                    result["added"][next_path] = right[key]
                elif key not in right:
                    result["removed"][next_path] = left[key]
                else:
                    walk(left[key], right[key], next_path)
            return

        if left != right:
            result["changed"][path] = (left, right)

    walk(before, after, ())
    return result


before = {
    "database": {"host": "old-db", "port": 5432},
    "features": {"search": True, "export": False},
    "version": 1,
}
after = {
    "database": {"host": "new-db", "port": 5432, "ssl": True},
    "features": {"search": True},
    "version": 2,
}

diff = dictionary_diff(before, after)

assert diff["added"] == {("database", "ssl"): True}
assert diff["removed"] == {("features", "export"): False}
assert diff["changed"] == {
    ("database", "host"): ("old-db", "new-db"),
    ("version",): (1, 2),
}

diff

{'added': {('database', 'ssl'): True},
 'removed': {('features', 'export'): False},
 'changed': {('database', 'host'): ('old-db', 'new-db'), ('version',): (1, 2)}}

**Best practice:** tuple paths avoid the escaping ambiguities of dotted strings and remain valid dictionary keys.

# Problem 20 — Convert a Tuple-Key Dictionary to JSON-Friendly Records

JSON object keys must be strings. A sparse matrix uses tuple keys, so direct JSON serialization is inappropriate.

Convert the mapping to a list of records and reconstruct it without information loss.

### Solution

In [25]:
def sparse_to_records(sparse: Mapping[tuple[int, int], float]) -> list[dict[str, float | int]]:
    return [
        {"row": row, "column": column, "value": value}
        for (row, column), value in sparse.items()
    ]


def records_to_sparse(records: Iterable[Mapping[str, Any]]) -> SparseMatrix:
    sparse: SparseMatrix = {}

    for position, record in enumerate(records):
        try:
            coordinate = (int(record["row"]), int(record["column"]))
            value = float(record["value"])
        except (KeyError, TypeError, ValueError) as exc:
            raise ValueError(f"Invalid sparse record at position {position}") from exc

        if coordinate in sparse:
            raise KeyError(f"Duplicate coordinate: {coordinate}")
        if value != 0:
            sparse[coordinate] = value

    return sparse


json_friendly_records = sparse_to_records(sparse)
restored_sparse = records_to_sparse(json_friendly_records)

assert restored_sparse == sparse
assert all(set(record) == {"row", "column", "value"} for record in json_friendly_records)

json_friendly_records

[{'row': 0, 'column': 1, 'value': 5.0},
 {'row': 1, 'column': 0, 'value': 2.0},
 {'row': 2, 'column': 2, 'value': 9.0}]

**Best practice:** design an explicit serialization schema instead of coercing complex keys to strings that may be difficult or unsafe to parse later.

# Final Review

You should now be able to:

- select the right dictionary-construction mechanism for fixed, paired, transformed, layered, or grouped data
- reason about hashability and convert nested mutable data into stable keys
- control duplicate-key behavior and merge precedence
- use functions and tuples as keys
- distinguish shallow copying from deep copying
- avoid the mutable-value `fromkeys` trap
- construct indexes, multimaps, nested aggregates, sparse matrices, tries, graphs, caches, and registries
- preserve insertion order intentionally
- validate inputs and write executable tests
- convert dictionaries with non-string keys into JSON-friendly schemas

A strong implementation is not merely concise: it makes invariants, ownership, collision behavior, and failure modes explicit.